<a href="https://colab.research.google.com/github/atrbyg24/kilter-board/blob/main/kilterboard_vit_convnext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import necessary libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, datasets, models
from PIL import Image
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np
import matplotlib.pyplot as plt
from transformers import ViTForImageClassification, AutoImageProcessor
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
import time
import shutil

In [2]:
# --- 1. Setup and Mount Google Drive ---

# Mount Google Drive (if your dataset is there)
from google.colab import drive
drive.mount('/content/drive')


DATA_ROOT = '/content/drive/MyDrive/Data/'
CHECKPOINT_DIR = '/content/drive/MyDrive/model_checkpoints/' # <--- NEW: DEFINE CHECKPOINT SAVE LOCATION
os.makedirs(CHECKPOINT_DIR, exist_ok=True) # Create the directory if it doesn't exist
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
print("-" * 50)


# Verify the path
if not os.path.exists(DATA_ROOT):
    print(f"Error: Dataset path '{DATA_ROOT}' does not exist. Please check your path and mount status.")
else:
    print(f"Dataset root found at: {DATA_ROOT}")

# Determine device for training (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be saved to: /content/drive/MyDrive/model_checkpoints/
--------------------------------------------------
Dataset root found at: /content/drive/MyDrive/Data/
Using device: cpu


In [3]:
# --- 2. Data Preparation ---

# Models like ViT-base-patch16-224 and ConvNeXt-base are typically trained on 224x224 images.
MODEL_INPUT_SIZE = (224, 224) # Standard input size for most pre-trained models

BATCH_SIZE = 32

# Normalization values for ImageNet pre-trained models
# These are standard for models pre-trained on ImageNet
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

# Data Augmentation and Normalization Transforms
train_transforms = transforms.Compose([
    transforms.Resize(MODEL_INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), # Converts PIL Image to PyTorch Tensor and scales to [0, 1]
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize(MODEL_INPUT_SIZE), # <--- Images will be resized here
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# Use ImageFolder to load data. It automatically infers labels from folder names.
# Initially, apply val_test_transforms as a placeholder, we'll reassign later for train/val/test splits.
full_dataset = datasets.ImageFolder(root=DATA_ROOT, transform=val_test_transforms)
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

print(f"Found {len(full_dataset)} images across {NUM_CLASSES} classes.")
print(f"Class names: {class_names}")

# Split the dataset into training, validation, and test sets
# Get indices for splitting, stratifying by target labels to maintain class distribution
train_idx, test_idx = train_test_split(
    list(range(len(full_dataset))),
    test_size=0.2, # 20% for test
    random_state=42,
    stratify=full_dataset.targets # Ensures even distribution of classes
)
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.25, # 25% of the remaining 80% (i.e., 20% of total) for validation
    random_state=42,
    stratify=[full_dataset.targets[i] for i in train_idx] # Stratify based on the subset of train_idx
)

# Create Subset datasets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

train_paths = [full_dataset.samples[i][0] for i in train_idx]
train_labels = [full_dataset.targets[i] for i in train_idx]
val_paths = [full_dataset.samples[i][0] for i in val_idx]
val_labels = [full_dataset.targets[i] for i in val_idx]
test_paths = [full_dataset.samples[i][0] for i in test_idx]
test_labels = [full_dataset.targets[i] for i in test_idx]

class CustomSubsetDataset(Dataset):
    """A wrapper to apply a specific transform to a list of (image_path, label) pairs."""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = CustomSubsetDataset(train_paths, train_labels, transform=train_transforms)
val_dataset = CustomSubsetDataset(val_paths, val_labels, transform=val_test_transforms)
test_dataset = CustomSubsetDataset(test_paths, test_labels, transform=val_test_transforms)


print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Create DataLoaders
# `num_workers=os.cpu_count()` is good for Colab to speed up data loading,
# but can be reduced if you hit memory issues.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=os.cpu_count())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count())



Found 12922 images across 8 classes.
Class names: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8']
Train samples: 7752
Validation samples: 2585
Test samples: 2585


In [4]:
# --- 3. Model Adaptation (Transfer Learning) & Training Loop ---

def save_checkpoint(model, optimizer, epoch, best_val_accuracy, history, filename, is_best):
    """Saves model checkpoint to disk."""
    state = {
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_val_accuracy': best_val_accuracy,
        'history': history # Save training history
    }
    filepath = os.path.join(CHECKPOINT_DIR, filename)
    torch.save(state, filepath)
    if is_best:
        best_filepath = os.path.join(CHECKPOINT_DIR, f'BEST_{filename}')
        shutil.copyfile(filepath, best_filepath)
        print(f"Saved BEST checkpoint to {best_filepath}")
    print(f"Saved checkpoint to {filepath}")

def load_checkpoint(model, optimizer, filename):
    """Loads model and optimizer state from a checkpoint."""
    filepath = os.path.join(CHECKPOINT_DIR, filename)
    if os.path.isfile(filepath):
        print(f"Loading checkpoint '{filename}'...")
        checkpoint = torch.load(filepath, map_location=device)
        model.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        start_epoch = checkpoint['epoch'] + 1 # Resume from the next epoch
        best_val_accuracy = checkpoint['best_val_accuracy']
        history = checkpoint['history']
        print(f"Checkpoint loaded. Resuming from epoch {start_epoch} with best_val_accuracy: {best_val_accuracy:.4f}")
        return model, optimizer, start_epoch, best_val_accuracy, history
    else:
        print(f"No checkpoint found at '{filepath}'. Starting fresh.")
        # Return initial values if no checkpoint is found
        return model, optimizer, 0, 0.0, {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}


def train_model(model, dataloader, criterion, optimizer, scheduler=None, num_epochs=10, val_dataloader=None, model_name="Model", checkpoint_filename_prefix="model"):
    print(f"\n--- Training {model_name} ---")
    start_time_overall = time.time() # To track total time even if resumed

    checkpoint_filename = f"{checkpoint_filename_prefix}_last.pth"
    model, optimizer, start_epoch, best_val_accuracy, history = load_checkpoint(model, optimizer, checkpoint_filename)

    for epoch in range(start_epoch, num_epochs):
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        start_epoch_time = time.time() # Track time per epoch

        for i, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            if isinstance(model, ViTForImageClassification):
                outputs = model(inputs).logits
            else:
                outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

        epoch_loss = running_loss / total_samples
        epoch_accuracy = correct_predictions / total_samples
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_accuracy)

        current_val_accuracy = 0.0 # Initialize for current epoch
        val_epoch_loss = 0.0 # Initialize for printing
        if val_dataloader:
            model.eval()
            val_running_loss = 0.0
            val_correct_predictions = 0
            val_total_samples = 0
            with torch.no_grad():
                for inputs, labels in val_dataloader:
                    inputs, labels = inputs.to(device), labels.to(device)

                    if isinstance(model, ViTForImageClassification):
                        outputs = model(inputs).logits
                    else:
                        outputs = model(inputs)

                    loss = criterion(outputs, labels)

                    val_running_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs.data, 1)
                    val_total_samples += labels.size(0)
                    val_correct_predictions += (predicted == labels).sum().item()

            val_epoch_loss = val_running_loss / val_total_samples
            val_epoch_accuracy = val_correct_predictions / val_total_samples
            history['val_loss'].append(val_epoch_loss)
            history['val_acc'].append(val_epoch_accuracy)
            current_val_accuracy = val_epoch_accuracy # Update for current epoch

        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f}, Train Accuracy: {epoch_accuracy:.4f} "
              f"| Val Loss: {val_epoch_loss:.4f}, Val Accuracy: {val_epoch_accuracy:.4f} "
              f"| Time: {(time.time() - start_epoch_time):.2f}s")

        is_best = current_val_accuracy > best_val_accuracy
        if is_best:
            best_val_accuracy = current_val_accuracy

        save_checkpoint(model, optimizer, epoch, best_val_accuracy, history, checkpoint_filename, is_best)

        if scheduler:
            scheduler.step()

    end_time_overall = time.time()
    print(f"Total training time for {model_name}: {(end_time_overall - start_time_overall):.2f} seconds")
    return model, history, best_val_accuracy

criterion = nn.CrossEntropyLoss()

#### **a. Vision Transformer (ViT) Model:**

# Load pre-trained ViT model from Hugging Face
model_name_vit = "google/vit-base-patch16-224"
model_vit = ViTForImageClassification.from_pretrained(
    model_name_vit,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

# Freeze all layers initially for feature extraction
for param in model_vit.parameters():
    param.requires_grad = False

# Only unfreeze the classification head
for param in model_vit.classifier.parameters():
    param.requires_grad = True

model_vit.to(device)
optimizer_vit_fe = AdamW(model_vit.parameters(), lr=1e-3)

print("\n--- ViT Model (Feature Extraction Phase) ---")
model_vit_fe, history_vit_fe, vit_fe_best_acc = train_model(
    model_vit, train_loader, criterion, optimizer_vit_fe,
    val_dataloader=val_loader, num_epochs=5, model_name="ViT (Feature Extraction)",
    checkpoint_filename_prefix="vit_fe"
)

# --- FINE-TUNING VIT: Unfreeze only the last few encoder layers ---
NUM_LAYERS_TO_UNFREEZE_VIT = 2 # Adjust this number as needed

# First, ensure all parameters are frozen again
for param in model_vit_fe.parameters():
    param.requires_grad = False

# Unfreeze the classifier
for param in model_vit_fe.classifier.parameters():
    param.requires_grad = True

# Accessing model_vit.vit.encoder.layer
for i, layer in enumerate(model_vit_fe.vit.encoder.layer):
    if i >= (len(model_vit_fe.vit.encoder.layer) - NUM_LAYERS_TO_UNFREEZE_VIT):
        for param in layer.parameters():
            param.requires_grad = True
        print(f"Unfrozen ViT encoder layer {i}")

# You might find LayerNorm layers within the encoder or after the pooling.
# For ViT, often the final LayerNorm before the classifier, and LayerNorms within the un-frozen blocks.
for name, param in model_vit_fe.named_parameters():
    if "layernorm" in name.lower() and param.requires_grad is False:
        param.requires_grad = True


# Create a new optimizer for the fine-tuning phase, only for trainable parameters
optimizer_vit_ft = AdamW(filter(lambda p: p.requires_grad, model_vit_fe.parameters()), lr=1e-5) # Lower LR
scheduler_vit_ft = StepLR(optimizer_vit_ft, step_size=5, gamma=0.1)

print(f"\n--- ViT Model (Fine-tuning Phase - Last {NUM_LAYERS_TO_UNFREEZE_VIT} layers + Classifier) ---")

model_vit_ft, history_vit_ft, vit_ft_best_acc = train_model(
    model_vit_fe, train_loader, criterion, optimizer_vit_ft, scheduler=scheduler_vit_ft,
    val_dataloader=val_loader, num_epochs=10, model_name="ViT (Fine-tuning)",
    checkpoint_filename_prefix="vit_ft"
)

#### **b. ConvNeXt Model:**

# Load pre-trained ConvNeXt model from torchvision
model_convnext = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)

# Freeze all layers initially
for param in model_convnext.parameters():
    param.requires_grad = False

# Replace the classification head
num_ftrs = model_convnext.classifier[-1].in_features
model_convnext.classifier[-1] = nn.Linear(num_ftrs, NUM_CLASSES)
model_convnext.to(device)

optimizer_convnext_fe = AdamW(model_convnext.parameters(), lr=1e-3)

print("\n--- ConvNeXt Model (Feature Extraction Phase) ---")
model_convnext_fe, history_convnext_fe, convnext_fe_best_acc = train_model(
    model_convnext, train_loader, criterion, optimizer_convnext_fe,
    val_dataloader=val_loader, num_epochs=5, model_name="ConvNeXt (Feature Extraction)",
    checkpoint_filename_prefix="convnext_fe"
)


# --- FINE-TUNING CONVNEXT: ---
# ConvNeXtBase has 4 "stages" in its features backbone.
# model.convnext_base.features is a Sequential module containing 4 blocks.
# Accessing them by index: model.convnext_base.features[0], [1], [2], [3]
# Stage 0 is the initial patch embedding and LayerNorm.
# Stage 1, 2, 3 are the main ConvNeXt blocks.

NUM_STAGES_TO_UNFREEZE_CONVNEXT = 1 # Adjust as needed.

# First, ensure all parameters are frozen again
for param in model_convnext_fe.parameters():
    param.requires_grad = False

# Unfreeze the classifier
for param in model_convnext_fe.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last N stages of the features backbone
total_stages = len(model_convnext_fe.features)
for i, stage in enumerate(model_convnext_fe.features):
    if i >= (total_stages - NUM_STAGES_TO_UNFREEZE_CONVNEXT):
        for param in stage.parameters():
            param.requires_grad = True
        print(f"Unfrozen ConvNeXt features stage {i}")

# Create a new optimizer for the fine-tuning phase, only for trainable parameters
optimizer_convnext_ft = AdamW(filter(lambda p: p.requires_grad, model_convnext_fe.parameters()), lr=1e-5) # Lower LR
scheduler_convnext_ft = StepLR(optimizer_convnext_ft, step_size=5, gamma=0.1)

print(f"\n--- ConvNeXt Model (Fine-tuning Phase - Last {NUM_STAGES_TO_UNFREEZE_CONVNEXT} stage(s) + Classifier) ---")
model_convnext_ft, history_convnext_ft, convnext_ft_best_acc = train_model(
    model_convnext_fe, train_loader, criterion, optimizer_convnext_ft, scheduler=scheduler_convnext_ft,
    val_dataloader=val_loader, num_epochs=10, model_name="ConvNeXt (Fine-tuning)",
    checkpoint_filename_prefix="convnext_ft"
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- ViT Model (Feature Extraction Phase) ---

--- Training ViT (Feature Extraction) ---
Loading checkpoint 'vit_fe_last.pth'...
Checkpoint loaded. Resuming from epoch 5 with best_val_accuracy: 0.2379
Total training time for ViT (Feature Extraction): 7.31 seconds
Unfrozen ViT encoder layer 10
Unfrozen ViT encoder layer 11

--- ViT Model (Fine-tuning Phase - Last 2 layers + Classifier) ---

--- Training ViT (Fine-tuning) ---
Loading checkpoint 'vit_ft_last.pth'...
Checkpoint loaded. Resuming from epoch 10 with best_val_accuracy: 0.3172
Total training time for ViT (Fine-tuning): 9.93 seconds


Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth
100%|██████████| 338M/338M [00:01<00:00, 310MB/s]



--- ConvNeXt Model (Feature Extraction Phase) ---

--- Training ConvNeXt (Feature Extraction) ---
Loading checkpoint 'convnext_fe_last.pth'...
Checkpoint loaded. Resuming from epoch 5 with best_val_accuracy: 0.2735
Total training time for ConvNeXt (Feature Extraction): 10.23 seconds
Unfrozen ConvNeXt features stage 7

--- ConvNeXt Model (Fine-tuning Phase - Last 1 stage(s) + Classifier) ---

--- Training ConvNeXt (Fine-tuning) ---
Loading checkpoint 'convnext_ft_last.pth'...
Checkpoint loaded. Resuming from epoch 10 with best_val_accuracy: 0.3002
Total training time for ConvNeXt (Fine-tuning): 15.63 seconds


In [ ]:
# --- 4. Evaluation ---

def evaluate_model(model, dataloader, criterion, class_names, model_name="Model"):
    print(f"\n--- {model_name} Model Evaluation ---")
    model.eval() # Set model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_true_labels = []

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            if isinstance(model, ViTForImageClassification):
                outputs = model(inputs).logits
            else:
                outputs = model(inputs)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

    test_loss = running_loss / total_samples
    test_accuracy = correct_predictions / total_samples
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

    print(f"\n{model_name} Classification Report:")
    # Ensure all_true_labels and all_predictions are lists/arrays of integers
    print(classification_report(all_true_labels, all_predictions, target_names=class_names))

    return test_loss, test_accuracy, all_true_labels, all_predictions

# Evaluate ViT
vit_test_loss, vit_test_accuracy, vit_true_labels, vit_predictions = evaluate_model(
    model_vit_ft, test_loader, criterion, class_names, model_name="ViT"
)

# Evaluate ConvNeXt
convnext_test_loss, convnext_test_accuracy, convnext_true_labels, convnext_predictions = evaluate_model(
    model_convnext_ft, test_loader, criterion, class_names, model_name="ConvNeXt"
)



--- ViT Model Evaluation ---
Test Loss: 1.5978
Test Accuracy: 0.3246

ViT Classification Report:
              precision    recall  f1-score   support

          V1       0.52      0.36      0.42       148
          V2       0.00      0.00      0.00       120
          V3       0.36      0.55      0.44       386
          V4       0.30      0.23      0.26       451
          V5       0.28      0.49      0.36       523
          V6       0.27      0.18      0.22       380
          V7       0.29      0.12      0.17       307
          V8       0.43      0.40      0.41       270

    accuracy                           0.32      2585
   macro avg       0.31      0.29      0.28      2585
weighted avg       0.31      0.32      0.30      2585


--- ConvNeXt Model Evaluation ---


In [ ]:
# --- 5. Comparative Analysis ---

def plot_training_history(history, title):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

print("\n--- Plotting Training Histories ---")
plot_training_history(history_vit_ft, "ViT Fine-tuning")
plot_training_history(history_convnext_ft, "ConvNeXt Fine-tuning")

print("\n--- Comparative Analysis Summary ---")
print(f"Final ViT Test Accuracy: {vit_test_accuracy:.4f}")
print(f"Final ConvNeXt Test Accuracy: {convnext_test_accuracy:.4f}")